# Introducción

El presente análisis estudia la base de datos de una aplicación de lectura que busca posicionarse en el mercado editorial post-pandemia. A través de consultas SQL sobre las tablas de libros, autores, editoriales, calificaciones y reseñas, se busca generar insights que sustenten una propuesta de valor para el producto.

# Preparación de los datos

Importamos nuestras librerias 

In [7]:
import pandas as pd
from sqlalchemy import create_engine

Nos conectamos a nuestra base de datos

In [8]:
db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

Verificamos la conexión 

In [9]:
query = "SELECT * FROM books LIMIT 5"
pd.io.sql.read_sql(query, con=engine)

,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


# Exploración de los datos

Mostramos las primeras 5 filas de cada tabla

In [ ]:
for table in ["books", "authors", "publishers", "ratings", "reviews"]:
    query = f"SELECT * FROM {table} LIMIT 5"
    print(f"\n--- {table} ---")
    display(pd.io.sql.read_sql(query, con=engine))


--- books ---


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268



--- authors ---


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd



--- publishers ---


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company



--- ratings ---


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2



--- reviews ---


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


- `books` — Contiene información sobre libros: título, identificador de autor, editorial, número de páginas y fecha de publicación.

- `authors` — Contiene el nombre y el identificador de cada autor.

- `publishers` — Contiene el nombre y el identificador de cada editorial.

- `ratings` — Contiene las calificaciones otorgadas por usuarios a libros específicos, con una escala numérica.

- `reviews` — Contiene reseñas de texto escritas por usuarios sobre libros específicos.

# ¿Cuantos libros han sido publicados después del primero de enero del año 2000?

In [ ]:
query_books_after_2000 = """

SELECT 
    COUNT(*) AS books_after_2000

FROM
    books
    
WHERE
    publication_date::date > '2000-01-01'::date;
"""

pd.io.sql.read_sql(query_books_after_2000, con=engine)

,books_after_2000
0,819


Se identificaron 819 libros publicados después del 1 de enero del 2000

# Número de reseñas de usuarios y la calificación promedio para cada libro.

In [17]:
query_books_reviews = """

SELECT
    books.title AS book_title,
    COUNT(DISTINCT reviews.review_id) AS review_count,
    AVG(ratings.rating) AS mean_rating
    
FROM
    books
    INNER JOIN reviews ON reviews.book_id = books.book_id
    INNER JOIN ratings ON ratings.book_id = books.book_id
    
GROUP BY
    book_title;
    
"""

pd.io.sql.read_sql(query_books_reviews, con=engine)

,book_title,review_count,mean_rating
0,'Salem's Lot,2,3.666667
1,1 000 Places to See Before You Die,1,2.500000
2,13 Little Blue Envelopes (Little Blue Envelope...,3,4.666667
3,1491: New Revelations of the Americas Before C...,2,4.500000
4,1776,4,4.000000
...,...,...,...
988,Wyrd Sisters (Discworld #6; Witches #2),3,3.666667
989,Xenocide (Ender's Saga #3),3,3.400000
990,Year of Wonders,4,3.200000
991,You Suck (A Love Story #2),2,4.500000


Se calculó el número de reseñas y la calificación promedio para 993 libros. Esta información permite a la plataforma identificar qué títulos generan mayor engagement entre los usuarios, combinando tanto la participación activa (reseñas escritas) como la valoración general (calificación promedio)

# ¿Que editorial ha publicado el mayor número de libros con más de 50 páginas?

In [19]:
query_publisher = """

SELECT
    publishers.publisher AS publisher,
    COUNT(books.book_id) AS books_published

FROM
    books
    INNER JOIN publishers ON publishers.publisher_id = books.publisher_id
    
WHERE
    books.num_pages > 50
    
GROUP BY
    publishers.publisher

ORDER BY
    books_published DESC
    
LIMIT
    1;
"""

pd.io.sql.read_sql(query_publisher, con=engine)

,publisher,books_published
0,Penguin Books,42


**Penguin Books** es la editorial con mayor número de libros publicados con más de 50 páginas, con un total de 42 títulos en el catálogo. Esta información es útil para identificar los socios editoriales más relevantes para la plataforma, ya que las editoriales con mayor catálogo representan potenciales alianzas estratégicas para ampliar la oferta de contenido.

# ¿Que autor tiene el libro mejor calificado? (Mas de 50 reseñas)

In [21]:
query_best_author = """

SELECT
    authors.author AS author,
    AVG(ratings.rating) AS avg_rating
    
FROM
    books
    INNER JOIN authors ON authors.author_id = books.author_id
    INNER JOIN ratings ON ratings.book_id = books.book_id

GROUP BY
    authors.author

HAVING
    COUNT(ratings.rating_id) > 50
    
ORDER BY
    avg_rating DESC
    
LIMIT
    1;
"""

pd.io.sql.read_sql(query_best_author, con=engine)

,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.288462


# Promedio de reseñas de texto de usuarios que calificaron mas de 50 libros

In [22]:
query_reviews = """

SELECT
    AVG(sub.count_reviews) AS avg_reviews

FROM
    (SELECT
        COUNT(review_id) as count_reviews
    
    FROM
        reviews
        
    WHERE
        username IN (
            SELECT username
            FROM ratings
            GROUP BY username
            HAVING COUNT(book_id) > 50
            )
            GROUP BY username
            ) AS sub;
"""


pd.io.sql.read_sql(query_reviews, con=engine)

,avg_reviews
0,24.333333


Los usuarios que han calificado más de 50 libros escriben en promedio 24.3 reseñas de texto, lo que evidencia un alto nivel de compromiso con la plataforma. Este segmento de usuarios altamente activos representa un activo valioso: sus reseñas generan contenido orgánico que puede influir en las decisiones de compra de otros usuarios. Se recomienda identificar y fidelizar a este segmento mediante programas de recompensas o acceso anticipado a nuevos títulos.

# Conclusión

El análisis de la base de datos de la plataforma de lectura reveló los siguientes hallazgos clave para desarrollar una propuesta de valor competitiva:
El catálogo cuenta con **819 libros** publicados después del año 2000, lo que garantiza una oferta de contenido contemporáneo relevante para los usuarios.

**Penguin Books** es la editorial con mayor presencia en el catálogo (*42 títulos con más de 50 páginas*), representando un socio editorial estratégico prioritario.

**J.K. Rowling y Mary GrandPré** son las autoras con mejor calificación promedio (*4.29*) entre autores con suficiente volumen de calificaciones, lo que las posiciona como referentes de calidad para la plataforma.

Finalmente, los **usuarios más activos** (*más de 50 calificaciones*) generan en **promedio 24.3 reseñas**, lo que sugiere una base de usuarios comprometidos capaces de generar contenido orgánico de valor.

Estos hallazgos sugieren que la plataforma tiene una base sólida de contenido y usuarios para competir en el mercado editorial digital, con oportunidades claras de crecimiento en alianzas editoriales y programas de fidelización de usuarios activos.